# Tale Role — QLoRA storyteller (7B stand-in)

Train **our** storyteller adapter. Same 7B recipe as mechanics. The 32B card waits for a larger GPU — not this Colab.

- Base: `Qwen/Qwen2.5-7B-Instruct`
- Data: `llm/datasets/literary/storyteller.jsonl` only. Do not load `synthetic/storyteller.jsonl` or `mechanics.jsonl`.
- Export only the LoRA adapter. Push to a **private** Hub repo `…/talerole-storyteller`.
- Narrate the engine result. Never invent dice, HP, or turn order. Do not require the total in the prose — the UI already shows it.
- Never paste player tables, emails, or OTP codes into this notebook.

Use a **new** Colab runtime. Do not continue a mechanics session.

In [ ]:
!pip install -q "transformers>=4.44" "datasets>=2.20" "peft>=0.12" "bitsandbytes>=0.43" "accelerate>=0.33" trl

Upload `llm/datasets/literary/storyteller.jsonl`. Do not upload the synthetic salad file, mechanics.jsonl, or production exports.

In [ ]:
from pathlib import Path
import json
from datasets import Dataset

path = Path("storyteller.jsonl")
assert path.exists(), "Upload llm/datasets/literary/storyteller.jsonl first"
rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

def to_text(row):
    user = json.dumps(row["input"], ensure_ascii=False)
    assistant = json.dumps(row["output"], ensure_ascii=False)
    return {
        "text": (
            "<|im_start|>system\nNarrate the engine result in the locale. Never invent dice, HP, or turn order. If the player stays put, they do not walk into a new place.<|im_end|>\n"
            f"<|im_start|>user\n{user}<|im_end|>\n"
            f"<|im_start|>assistant\n{assistant}<|im_end|>\n"
        )
    }

ds = Dataset.from_list([to_text(r) for r in rows])
print(len(ds), "rows")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

base = "Qwen/Qwen2.5-7B-Instruct"
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)
tok = AutoTokenizer.from_pretrained(base, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(
    base, quantization_config=bnb, device_map="auto", trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)
model = get_peft_model(
    model,
    LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM", target_modules="all-linear"),
)
args = SFTConfig(
    output_dir="storyteller-qlora",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=1,
    fp16=False,
    bf16=use_bf16,
    report_to=[],
    dataset_text_field="text",
    max_length=1024,
)
trainer = SFTTrainer(model=model, processing_class=tok, train_dataset=ds, args=args)
trainer.train()
trainer.save_model("storyteller-adapter")
tok.save_pretrained("storyteller-adapter")
print("adapter saved")

In [ ]:
from huggingface_hub import login, HfApi

login()  # paste write token in the prompt — not in git, not in chat
repo = "levonov/talerole-storyteller"
HfApi().create_repo(repo, private=True, exist_ok=True)
HfApi().upload_folder(folder_path="storyteller-adapter", repo_id=repo)
print("Private Hub is the weight store. GPU runner: HF_MODEL_ID. Never git.")